In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression


# ============================================================
# 1. Load data
# ============================================================

# CSV file name
file_name = "data8_6p2Hs_8Tp.csv"

# Load the CSV file without headers
data = pd.read_csv(file_name, header=None)


# ============================================================
# 2. Extract relevant signals
# ============================================================

# Input signal
excitation1 = data.iloc[:, 0].values

# Output signals
vpto = data.iloc[:, 2].values        # Output 1: VPTO / velocity
position = data.iloc[:, 3].values    # Output 2: Position


# ============================================================
# 3. Select training data range
# ============================================================

# Data interval used for the mutual information analysis
start_index = 2000
end_index = 3000

train_excitation1 = excitation1[start_index:end_index]
train_vpto = vpto[start_index:end_index]
train_position = position[start_index:end_index]


# ============================================================
# 4. Mutual information function
# ============================================================

def mutual_information_analysis(input_signal, output_signal, max_lag=100):
    """
    Compute the mutual information between an input signal and an output signal
    for different positive time lags.

    Parameters
    ----------
    input_signal : array-like
        Signal used as the input variable.
        
    output_signal : array-like
        Signal used as the output variable.
        
    max_lag : int, optional
        Maximum lag to be evaluated. Default is 100.

    Returns
    -------
    lags : ndarray
        Array containing the evaluated lag values.
        
    mi_values : ndarray
        Mutual information value for each lag.
    """

    lags = np.arange(1, max_lag + 1)
    mi_values = []

    for lag in lags:
        # Shift the input and output signals according to the current lag
        shifted_input = input_signal[:-lag]
        shifted_output = output_signal[lag:]

        # Compute mutual information
        mi = mutual_info_regression(
            shifted_input.reshape(-1, 1),
            shifted_output
        )

        mi_values.append(mi[0])

    return lags, np.array(mi_values)


# ============================================================
# 5. Compute mutual information for VPTO
# ============================================================

# Mutual information between VPTO and its own past values
t_lags_vpto, mi_ylag_vpto = mutual_information_analysis(
    train_vpto,
    train_vpto
)

# Mutual information between Excitation1 and VPTO
t_lags_xlag_vpto, mi_xlag_vpto = mutual_information_analysis(
    train_excitation1,
    train_vpto
)


# ============================================================
# 6. Compute mutual information for Position
# ============================================================

# Mutual information between Position and its own past values
t_lags_position, mi_ylag_position = mutual_information_analysis(
    train_position,
    train_position
)

# Mutual information between Excitation1 and Position
t_lags_xlag_position, mi_xlag_position = mutual_information_analysis(
    train_excitation1,
    train_position
)


# ============================================================
# 7. Identify the most relevant lags
# ============================================================

# Percentile used as the threshold for selecting significant lags
percent_max = 90

# VPTO: significant x-lags
threshold_vpto = np.percentile(mi_xlag_vpto, percent_max)
significant_xlags_vpto = t_lags_xlag_vpto[mi_xlag_vpto > threshold_vpto]

# VPTO: significant y-lags
y_threshold_vpto = np.percentile(mi_ylag_vpto, percent_max)
significant_ylags_vpto = t_lags_vpto[mi_ylag_vpto > y_threshold_vpto]

# Position: significant x-lags
threshold_position = np.percentile(mi_xlag_position, percent_max)
significant_xlags_position = t_lags_xlag_position[
    mi_xlag_position > threshold_position
]

# Position: significant y-lags
y_threshold_position = np.percentile(mi_ylag_position, percent_max)
significant_ylags_position = t_lags_position[
    mi_ylag_position > y_threshold_position
]


# ============================================================
# 8. Print relevant lags
# ============================================================

print("Most relevant lags for VPTO - y-lag:", significant_ylags_vpto)
print("Most relevant lags for VPTO - x-lag:", significant_xlags_vpto)

print("Most relevant lags for Position - y-lag:", significant_ylags_position)
print("Most relevant lags for Position - x-lag:", significant_xlags_position)


# ============================================================
# 9. Configure plot style
# ============================================================

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 18
})


# ============================================================
# 10. Plot mutual information results
# ============================================================

plt.figure(figsize=(12, 10))


# ------------------------------------------------------------
# VPTO y-lag
# ------------------------------------------------------------

plt.subplot(2, 2, 1)

plt.plot(
    t_lags_vpto,
    mi_ylag_vpto,
    marker="o",
    color="black"
)

plt.scatter(
    significant_ylags_vpto,
    mi_ylag_vpto[mi_ylag_vpto > y_threshold_vpto],
    color="red",
    zorder=3
)

plt.axhline(y=0, color="black", linestyle="--")
plt.xlabel("y-Lag")
plt.ylabel("Mutual Information")
plt.title("Velocity")


# ------------------------------------------------------------
# VPTO x-lag
# ------------------------------------------------------------

plt.subplot(2, 2, 2)

plt.plot(
    t_lags_xlag_vpto,
    mi_xlag_vpto,
    marker="o",
    color="black"
)

plt.scatter(
    significant_xlags_vpto,
    mi_xlag_vpto[mi_xlag_vpto > threshold_vpto],
    color="red",
    zorder=3
)

plt.axhline(y=0, color="black", linestyle="--")
plt.xlabel("x-Lag")
plt.ylabel("Mutual Information")
plt.title("Velocity")


# ------------------------------------------------------------
# Position y-lag
# ------------------------------------------------------------

plt.subplot(2, 2, 3)

plt.plot(
    t_lags_position,
    mi_ylag_position,
    marker="o",
    color="black"
)

plt.scatter(
    significant_ylags_position,
    mi_ylag_position[mi_ylag_position > y_threshold_position],
    color="red",
    zorder=3
)

plt.axhline(y=0, color="black", linestyle="--")
plt.xlabel("y-Lag")
plt.ylabel("Mutual Information")
plt.title("Position")


# ------------------------------------------------------------
# Position x-lag
# ------------------------------------------------------------

plt.subplot(2, 2, 4)

plt.plot(
    t_lags_xlag_position,
    mi_xlag_position,
    marker="o",
    color="black"
)

plt.scatter(
    significant_xlags_position,
    mi_xlag_position[mi_xlag_position > threshold_position],
    color="red",
    zorder=3
)

plt.axhline(y=0, color="black", linestyle="--")
plt.xlabel("x-Lag")
plt.ylabel("Mutual Information")
plt.title("Position")


# ============================================================
# 11. Save and display figure
# ============================================================

plt.tight_layout()
plt.savefig("MI.svg", format="svg")
plt.show()